# EAHN — Specialist Training Notebook (Phase 17)

**Workflow:** fork this notebook five times on Kaggle. In each fork, edit **`ACTIVE_MANIPULATION`** in Cell 2 to one of:

`Deepfakes` · `Face2Face` · `FaceShifter` · `FaceSwap` · `NeuralTextures`

Each fork trains a specialist on 1000 real + 1000 of that one manipulation, then evaluates on:
1. The held-out FF++ test split (100 real + 100 of that manipulation)
2. The Celeb-DF v2 official test split (~178 real + ~340 fake)

Run every cell **top to bottom**. No inline patches — all logic is in the repo.

| Cell | Purpose | Gate before next |
|------|---------|-----------------|
| 1 | Auto-detect paths + set ACTIVE_MANIPULATION | All three paths print correctly |
| 2 | File count check (FF++ + Celeb-DF) | All directories present |
| 3 | Clean previous run | — |
| 4 | Clone repo from GitHub | Repo at Phase 17 commit |
| 5 | Install deps + import check | ALL IMPORTS OK |
| 6 | Specialist dataset verification | SPECIALIST VERIFIED |
| 6b | (Optional) restore from uploaded checkpoint | Skip Cell 7 if Restore complete |
| 7  | Train + Evaluate (FF++ test + Celeb-DF test + explanation suite) | Val AUC-ROC rises above 0.6 by epoch 5 |
| 7c | Collapse diagnostic check | No collapse detected |
| 8 | Metrics table (FF++ + Celeb-DF + explanation) | — |
| 9 | Detection graphs (ROC, PR, confusion, score dist) | — |
| 10 | Heatmap overlays (intrinsic, Grad-CAM, Att-Rollout) | — |
| 11 | Zip for download (named with active manipulation) | — |


---
## Cell 1 — Auto-detect paths + configuration

In [ ]:
import os, glob

# ════════════════════════════════════════════════════════════════════════════
# ║                EDIT THESE FOR THIS NOTEBOOK FORK                          ║
# ║   One specialist per fork. Change ACTIVE_MANIPULATION between forks.      ║
# ════════════════════════════════════════════════════════════════════════════

ACTIVE_MANIPULATION = "Deepfakes"
#                     ^^^^^^^^^^
#  one of: "Deepfakes" | "Face2Face" | "FaceShifter" | "FaceSwap" | "NeuralTextures"

FFPP_DATA_ROOT    = "/kaggle/input/datasets/umardrazbhatti/ffpp-c23-custom-layout/ffpp_data"
CELEBDF_DATA_ROOT = "/kaggle/input/datasets/reubensuju/celeb-df-v2"

EPOCHS     = 19      # ~target for a single Kaggle session at T=16 / batch=4
BATCH_SIZE = 4

# ════════════════════════════════════════════════════════════════════════════

# Validate ACTIVE_MANIPULATION
_VALID = {"Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"}
if ACTIVE_MANIPULATION not in _VALID:
    raise SystemExit(
        f"ACTIVE_MANIPULATION='{ACTIVE_MANIPULATION}' is invalid. "
        f"Must be one of: {sorted(_VALID)}"
    )

# Fallback: auto-detect FF++ if hardcoded path missing
def _find_ffpp_root(base="/kaggle/input", max_depth=6):
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        if (os.path.isdir(os.path.join(root, "original_sequences")) and
                os.path.isdir(os.path.join(root, "manipulated_sequences"))):
            return root
    return None

if not os.path.isdir(FFPP_DATA_ROOT):
    print(f"[WARN] FFPP_DATA_ROOT not found at hardcoded path: {FFPP_DATA_ROOT}")
    print(f"       Attempting auto-detect under /kaggle/input ...")
    fallback = _find_ffpp_root()
    if fallback:
        FFPP_DATA_ROOT = fallback
        print(f"       Found: {FFPP_DATA_ROOT}")
    else:
        print("[ERROR] FF++ root not found. Available inputs:")
        for e in sorted(glob.glob("/kaggle/input/*/*"))[:30]:
            print(f"   {e}")
        raise SystemExit("Fix FFPP_DATA_ROOT in Cell 1.")

# Derived paths
DATA_ROOT   = FFPP_DATA_ROOT          # legacy name kept for other cells
REPO_URL    = "https://github.com/umardrazbhatti/EahnCode.git"
REPO_DIR    = "/kaggle/working/EahnCode"
OUTPUT_DIR  = "/kaggle/working/outputs"
CACHE_DIR   = "/kaggle/working/.face_cache"

print("=" * 65)
print(f"ACTIVE_MANIPULATION : {ACTIVE_MANIPULATION}")
print(f"FFPP_DATA_ROOT      : {FFPP_DATA_ROOT}")
print(f"CELEBDF_DATA_ROOT   : {CELEBDF_DATA_ROOT}")
print(f"REPO_DIR            : {REPO_DIR}")
print(f"OUTPUT_DIR          : {OUTPUT_DIR}")
print(f"EPOCHS              : {EPOCHS}")
print(f"BATCH_SIZE          : {BATCH_SIZE}")
print("=" * 65)
print(f"This fork trains on: 1000 real + 1000 {ACTIVE_MANIPULATION}")
print(f"This fork tests on : FF++ test split + Celeb-DF v2 official test split")


---
## Cell 2 — File count sanity check
Every row must show **OK** before continuing.

In [ ]:
import glob, os

METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]

# ── FF++ check ───────────────────────────────────────────────────────────────
real_dir  = os.path.join(FFPP_DATA_ROOT, "original_sequences", "youtube", "c23", "videos")
real_vids = glob.glob(os.path.join(real_dir, "*.mp4"))
status    = "OK" if len(real_vids) > 0 else "MISSING"
print(f"[FF++ {status:^7}]  {'original (real)':<22}  {len(real_vids):>5} videos")

active_count, all_ok = 0, len(real_vids) > 0
for method in METHODS:
    d      = os.path.join(FFPP_DATA_ROOT, "manipulated_sequences", method, "c23", "videos")
    count  = len(glob.glob(os.path.join(d, "*.mp4")))
    status = "OK" if count > 0 else "MISSING"
    marker = "  <--- ACTIVE" if method == ACTIVE_MANIPULATION else ""
    if method == ACTIVE_MANIPULATION:
        active_count = count
    if count == 0 and method == ACTIVE_MANIPULATION:
        all_ok = False
    print(f"[FF++ {status:^7}]  {method:<22}  {count:>5} videos{marker}")

print(f"\n[FF++] real={len(real_vids)}  active({ACTIVE_MANIPULATION})={active_count}")

if not all_ok or active_count == 0:
    raise SystemExit(
        f"FF++ data missing for ACTIVE_MANIPULATION={ACTIVE_MANIPULATION}. "
        f"Fix FFPP_DATA_ROOT in Cell 1."
    )

# ── Celeb-DF v2 check ────────────────────────────────────────────────────────
print()
celebdf_real_dir   = os.path.join(CELEBDF_DATA_ROOT, "Celeb-real")
celebdf_yt_dir     = os.path.join(CELEBDF_DATA_ROOT, "YouTube-real")
celebdf_synth_dir  = os.path.join(CELEBDF_DATA_ROOT, "Celeb-synthesis")
celebdf_list_path  = os.path.join(CELEBDF_DATA_ROOT, "List_of_testing_videos.txt")

n_celeb_real  = len(glob.glob(os.path.join(celebdf_real_dir, "*.mp4")))
n_yt_real     = len(glob.glob(os.path.join(celebdf_yt_dir, "*.mp4")))
n_synth       = len(glob.glob(os.path.join(celebdf_synth_dir, "*.mp4")))
has_test_list = os.path.isfile(celebdf_list_path)

for label, path, n in [
    ("Celeb-real",       celebdf_real_dir,  n_celeb_real),
    ("YouTube-real",     celebdf_yt_dir,    n_yt_real),
    ("Celeb-synthesis",  celebdf_synth_dir, n_synth),
]:
    status = "OK" if n > 0 else "MISSING"
    print(f"[Celeb-DF {status:^7}]  {label:<22}  {n:>5} videos")

list_status = "OK" if has_test_list else "MISSING"
print(f"[Celeb-DF {list_status:^7}]  List_of_testing_videos.txt")

if not has_test_list:
    raise SystemExit(
        f"Celeb-DF test list not found at {celebdf_list_path}. "
        f"Fix CELEBDF_DATA_ROOT in Cell 1, or attach the Celeb-DF v2 Kaggle Dataset."
    )

# Peek test list for expected ~518 entries
with open(celebdf_list_path) as f:
    n_test_entries = sum(1 for line in f if line.strip())
print(f"[Celeb-DF        ]  test entries  : {n_test_entries}  (expected ~518)")

print("\nAll dataset checks passed - proceed to Cell 3.")


---
## Cell 3 — Clean previous run
Deletes old clone, outputs and cache. Dataset under `/kaggle/input/` is never touched.

In [ ]:
import shutil, os

to_remove = [REPO_DIR, OUTPUT_DIR, "/kaggle/working/eahn_outputs.zip"]
for path in to_remove:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir  : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file : {path}")
    else:
        print(f"Not present  : {path}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

fc = "/kaggle/working/.face_cache"
if os.path.isdir(fc):
    n_cached = sum(len(files) for _, _, files in os.walk(fc))
    print(f"[face_cache] preserved: {n_cached} cached files at {fc}")
else:
    os.makedirs(fc, exist_ok=True)
    print(f"[face_cache] empty cache created at {fc} (first run will be slow)")

print()
print("Clean. Proceed to Cell 4.")

---
## Cell 4 — Clone repo from GitHub
⚠️ **Internet must be ON**: Kaggle right panel → Settings → Internet → On.

In [ ]:
import os, sys, subprocess

ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")

if ret != 0 or not os.path.isdir(REPO_DIR):
    raise SystemExit(
        "Clone failed.\n"
        "Fix: Enable Internet in Kaggle Settings (right panel → Internet → On).\n"
        "Then re-run this cell."
    )

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR).decode().strip()
commit = subprocess.check_output(["git", "log", "--oneline", "-1"],   cwd=REPO_DIR).decode().strip()
print(f"\nClone successful.")
print(f"  Branch : {branch}")
print(f"  Commit : {commit}")
print("\nRepo contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(f"  {f}")


---
## Cell 5 — Install dependencies + full import chain check
Must print **ALL IMPORTS OK** before continuing.

In [ ]:
import subprocess, sys, importlib, os

# ── Ensure REPO_DIR is first on sys.path and CWD ─────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Flush stale module cache (guards against re-running without restarting kernel)
stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["config", "data", "models", "losses", "xai", "metrics", "utils", "scripts"])]
for k in stale:
    del sys.modules[k]

print("Installing requirements...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     f"{REPO_DIR}/requirements.txt"],
    check=True
)
print("Installation complete.\n")

# ── Key package versions ─────────────────────────────────────────────────────
pkg_map = {"torch":"torch","torchvision":"torchvision","timm":"timm",
           "sklearn":"sklearn","cv2":"cv2","PIL":"PIL","tqdm":"tqdm"}
for name, mod_name in pkg_map.items():
    try:
        m = importlib.import_module(mod_name)
        print(f"  OK      {name:<15} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING {name}")

# ── Full import chain ─────────────────────────────────────────────────────────
print("\nChecking full import chain...")
try:
    from config import EAHNConfig                            # noqa
    from data.datasets import DeepfakeDataset                # noqa
    from data.collate import deepfake_collate_fn             # noqa
    from data.transforms import get_transforms               # noqa
    from data.face_align import FaceAligner                  # noqa
    from data.synthetic_generator import SyntheticDataGenerator  # noqa
    from models.eahn import EAHN                             # noqa
    from losses.classification import ClassificationLoss     # noqa
    from losses.explanation import ExplanationLoss           # noqa
    from losses.temporal import TemporalConsistencyLoss      # noqa
    from metrics.detection import DetectionMetrics           # noqa
    from scripts.train_real import main as train_main        # noqa
    from scripts.evaluate import run_evaluation              # noqa
    from scripts.dashboard import show_dashboard             # noqa
    print("\nALL IMPORTS OK — proceed to Cell 6.")
except ImportError as e:
    raise SystemExit(
        f"\nIMPORT FAILED: {e}\n"
        "The repo fix (Claude Code prompt) has not been pushed yet, or the "
        "clone in Cell 4 pulled an old commit.\n"
        "Action: re-run from Cell 3 to get a fresh clone."
    )

# ── Verify config params ──────────────────────────────────────────────────────
# Use max_per_class=1000 to match Cell 7 CLI flags (Regime A).
config = EAHNConfig(max_per_class=1000)
assert hasattr(config, 'attn_temp_init'),        "attn_temp_init missing from EAHNConfig"
assert hasattr(config, 'attn_diversity_weight'), "attn_diversity_weight missing from EAHNConfig"
assert config.gamma == 0.1,             f"gamma must be 0.1, got {config.gamma}"
assert config.cls_loss_type == "bce",   f"cls_loss_type must be 'bce', got {config.cls_loss_type!r}"
assert config.cls_dropout_p == 0.0,    f"cls_dropout_p must be 0.0, got {config.cls_dropout_p}"
assert config.batch_size    == 4,      f"batch_size must be 4, got {config.batch_size}"
assert config.grad_accum_steps == 4,   f"grad_accum_steps must be 4, got {config.grad_accum_steps}"
assert getattr(config, 'use_amp', None) is True, "use_amp must be True"
assert getattr(config, 'grad_checkpoint', None) is True, "grad_checkpoint must be True"
assert getattr(config, 'focal_alpha', None) == 1.0, "focal_alpha missing or != 1.0"
assert getattr(config, 'focal_gamma', None) == 1.0,  "focal_gamma missing or != 1.0"
assert hasattr(config, "label_smoothing"),    "label_smoothing missing from EAHNConfig"
assert hasattr(config, "max_per_class"),       "max_per_class missing from EAHNConfig"

# Regime A sanity check
assert config.max_per_class == 1000, (
    f"Regime A expects max_per_class=1000, got {config.max_per_class}"
)

print("Config verified:")
print(f"  gamma            = {config.gamma}")
print(f"  attn_temp_init   = {config.attn_temp_init}")
print(f"  diversity_w      = {config.attn_diversity_weight}")
print(f"  cls_loss_type    = {config.cls_loss_type}")
print(f"  cls_dropout_p    = {config.cls_dropout_p}")
print(f"  batch_size       = {config.batch_size}  (effective={config.batch_size * config.grad_accum_steps})  AMP={config.use_amp}  GradCkpt={config.grad_checkpoint}")
print(f"  grad_accum_steps = {config.grad_accum_steps}")
print(f"  use_amp          = {config.use_amp}")
print(f"  grad_checkpoint  = {config.grad_checkpoint}")
print(f"  focal_alpha      = {config.focal_alpha}")
print(f"  focal_gamma      = {config.focal_gamma}")
print(f"  max_per_class    = {config.max_per_class}  [Regime A: 1000:1000 balanced]")

---
## Cell 6 — Dataset loader verification ✅
Loads one real training batch and asserts both classes are present.
**Do not run Cell 7 unless this prints VERIFICATION PASSED.**


In [ ]:
import os, sys, torch
from collections import Counter
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Flush stale module cache
stale = [k for k in list(sys.modules) if any(k.startswith(p) for p in
         ["data","models","losses","xai","metrics","utils","scripts","config"])]
for k in stale: del sys.modules[k]

from config import EAHNConfig
from data.datasets import DeepfakeDataset
from data.collate  import deepfake_collate_fn
from torch.utils.data import DataLoader

# Specialist config (num_frames=4 for verification only; training uses 16)
_cfg = EAHNConfig(
    data_root           = FFPP_DATA_ROOT,
    output_dir          = OUTPUT_DIR,
    cache_dir           = CACHE_DIR,
    dataset_name        = "ff++",
    num_frames          = 4,
    frame_size          = 224,
    train_split         = 0.8,
    val_split           = 0.1,
    device              = "auto",
    num_workers         = 0,
    max_per_class       = 1000,
    active_manipulation = ACTIVE_MANIPULATION,
)

print(f"Building specialist dataset (active={ACTIVE_MANIPULATION}, "
      f"max_per_class=1000) ...")
ds = DeepfakeDataset(_cfg, mode="train", dataset_type="ff++")

# ── Specialist-mode assertions ────────────────────────────────────────────────
labels = [s["label"] for s in ds.samples]
n_real = labels.count(0)
n_fake = labels.count(1)

fake_types = Counter()
for s in ds.samples:
    if s["label"] == 1:
        fake_types[s.get("manipulation")] += 1

print(f"  Real : {n_real}")
print(f"  Fake : {n_fake}")
print(f"  Per-fake-type: {dict(fake_types)}")

# Assertion 1: balance (1:1)
assert abs(n_real - n_fake) <= 5, (
    f"SPECIALIST VIOLATION: train set not balanced. "
    f"real={n_real}  fake={n_fake}  diff={abs(n_real-n_fake)}"
)
# Assertion 2: exactly ONE fake type present
assert len(fake_types) == 1, (
    f"SPECIALIST VIOLATION: expected only {ACTIVE_MANIPULATION} in fakes, "
    f"got {dict(fake_types)}"
)
# Assertion 3: that one type IS the active one
assert ACTIVE_MANIPULATION in fake_types, (
    f"SPECIALIST VIOLATION: {ACTIVE_MANIPULATION} not present; "
    f"got {list(fake_types.keys())}"
)

print(f"SPECIALIST VERIFIED: train={n_real+n_fake} "
      f"real={n_real} fake={n_fake} ({ACTIVE_MANIPULATION} only)")

# ── Batch check — plain shuffle loader (no sampler at 1:1) ───────────────────
loader = DataLoader(ds, batch_size=8, shuffle=True,
                    collate_fn=deepfake_collate_fn, num_workers=0)

print("\nInspecting 3 batches (shuffle=True):")
saw_real = saw_fake = False
for i, b in enumerate(loader):
    u = b["label"].unique().tolist()
    r = int((b["label"] == 0).sum()); f = int((b["label"] == 1).sum())
    print(f"  batch {i}: real={r} fake={f} unique={u}")
    if r > 0: saw_real = True
    if f > 0: saw_fake = True
    if i == 2: break

assert saw_real and saw_fake, "Loader not delivering both classes across 3 batches."
print()
print("=" * 65)
print(f"VERIFICATION PASSED - specialist mode on {ACTIVE_MANIPULATION}.")
print("Proceed to Cell 7.")
print("=" * 65)


---
## Cell 6b — Restore from uploaded checkpoint (optional)

If you have a previous `best_model.pth` uploaded as a private Kaggle Dataset, this cell copies it (and the face cache, if present) back into `OUTPUT_DIR` / `CACHE_DIR`. Lets you **skip Cell 7 (training)** and jump straight to **Cell 7b (per-manipulation breakdown re-eval).**

**Setup (one-time on Kaggle):**
1. Locally, zip your downloaded outputs containing at minimum `best_model.pth`. Recommended additions: the `.face_cache/` directory (saves ~3–5 min on eval) and `training_history.csv` / `logs.csv` (preserves loss curves for graphs).
2. Create a private Kaggle Dataset from the zip: https://www.kaggle.com/datasets → New Dataset → Upload.
3. Attach the dataset to this notebook: right panel → **Add Data** → **Your Datasets** → select your dataset.
4. Re-run this cell. The finder auto-detects any directory under `/kaggle/input/` containing `best_model.pth`.

If no upload is found, the cell prints instructions and exits cleanly — you can then run Cell 7 as usual to train from scratch.

In [ ]:
import os, shutil

def _find_uploaded_checkpoint(base="/kaggle/input", max_depth=3):
    """Find a Kaggle Dataset directory containing best_model.pth."""
    if not os.path.isdir(base):
        return None
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        if "best_model.pth" in files:
            return root
    return None

upload_dir = _find_uploaded_checkpoint()

if upload_dir is None:
    print("[INFO] No uploaded checkpoint found under /kaggle/input/.")
    print()
    print("To skip Cell 7 (training) and reuse a previous run:")
    print("  1. Zip your local outputs containing best_model.pth")
    print("     (and optionally the .face_cache/ directory).")
    print("  2. Create a private Kaggle Dataset from that zip.")
    print("  3. Attach the dataset to this notebook:")
    print("     right panel -> Add Data -> Your Datasets.")
    print("  4. Re-run this cell.")
    print()
    print("Without an upload, run Cell 7 (training) before Cell 7b.")
else:
    print(f"[INFO] Found uploaded checkpoint at: {upload_dir}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Required: best_model.pth
    src_ckpt = os.path.join(upload_dir, "best_model.pth")
    dst_ckpt = os.path.join(OUTPUT_DIR, "best_model.pth")
    shutil.copy2(src_ckpt, dst_ckpt)
    size_mb = os.path.getsize(dst_ckpt) / (1024 * 1024)
    print(f"  Copied best_model.pth ({size_mb:.1f} MB) -> {dst_ckpt}")

    # Optional: face cache (massive speedup for eval)
    src_cache = os.path.join(upload_dir, ".face_cache")
    if os.path.isdir(src_cache):
        os.makedirs(CACHE_DIR, exist_ok=True)
        n_copied = 0
        for item in os.listdir(src_cache):
            s = os.path.join(src_cache, item)
            d = os.path.join(CACHE_DIR, item)
            if os.path.isdir(s):
                if os.path.isdir(d):
                    shutil.rmtree(d)
                shutil.copytree(s, d)
                n_copied += sum(len(files) for _, _, files in os.walk(d))
            else:
                shutil.copy2(s, d)
                n_copied += 1
        print(f"  Copied face cache ({n_copied} files) -> {CACHE_DIR}")
    else:
        print("  [INFO] No .face_cache/ in upload — Cell 7b will rebuild"
              " face crops via MTCNN (adds ~3–5 min on T4).")

    # Optional auxiliary files for notebook UI continuity
    aux_files = ["training_history.csv", "logs.csv", "metrics.csv",
                 "metrics.json", "loss_curves.png", "metric_curves.png"]
    n_aux = 0
    for aux in aux_files:
        src_aux = os.path.join(upload_dir, aux)
        if os.path.isfile(src_aux):
            shutil.copy2(src_aux, os.path.join(OUTPUT_DIR, aux))
            n_aux += 1
    if n_aux:
        print(f"  Copied {n_aux} auxiliary file(s) (history/logs/metrics).")

    print()
    print("=" * 60)
    print("Restore complete.")
    print("You can now SKIP Cell 7 (training) and run Cell 7b directly.")
    print("=" * 60)


---
## Cell 7 — Train + Evaluate
Expected: ~4–6 min per epoch on Tesla T4.
After epoch 1 the AUC-ROC should be > 0.5, confirming the model is learning.
The face cache is cleared here first to prevent the T=4 (Cell 6) vs T=16
(training) frame-count collision that caused a RuntimeError in earlier runs.


In [ ]:
import os, sys, shutil
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Safeguard: do not overwrite a restored checkpoint
_existing_ckpt = os.path.join(OUTPUT_DIR, "best_model.pth")
if os.path.isfile(_existing_ckpt):
    _size_mb = os.path.getsize(_existing_ckpt) / (1024 * 1024)
    print("=" * 65)
    print(f"[SKIP] best_model.pth already exists ({_size_mb:.1f} MB) at:")
    print(f"   {_existing_ckpt}")
    print()
    print("Cell 6b likely restored a previous checkpoint.")
    print("Training is SKIPPED to avoid overwriting it.")
    print()
    print("To train from scratch, delete the file first:")
    print(f"   import os; os.remove({_existing_ckpt!r})")
    print("=" * 65)
else:
    # Clear face cache to prevent T=4 (Cell 6) vs T=16 (training) frame-count
    # collision on the same video_id keys
    if os.path.isdir(CACHE_DIR):
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR, exist_ok=True)
        print("Face cache cleared.")

    # Phase 17 specialist launcher
    #   - --active_manipulation : the one fake type to specialize on
    #   - --celebdf_root + --celebdf_eval : run Celeb-DF v2 test after FF++ test
    #   - --cls_loss_type bce : matches thesis spec; 1:1 balance => no focal needed
    #   - --explanation_suite (config default ON) : runs frame-drop + stability
    #   - --save_last_checkpoint NOT passed : default OFF, single-session training
    #   - batch_size=4, grad_accum_steps=4 -> effective batch=16 (T4 14.56 GiB safe)
    cmd = (
        f'python run_full_pipeline.py'
        f' --data_root          "{FFPP_DATA_ROOT}"'
        f' --dataset_name       ff++'
        f' --output_dir         "{OUTPUT_DIR}"'
        f' --cache_dir          "{CACHE_DIR}"'
        f' --active_manipulation {ACTIVE_MANIPULATION}'
        f' --celebdf_root       "{CELEBDF_DATA_ROOT}"'
        f' --celebdf_eval'
        f' --epochs             {EPOCHS}'
        f' --batch_size         {BATCH_SIZE}'
        f' --num_workers        2'
        f' --grad_accum_steps   4'
        f' --gamma              0.1'
        f' --attn_temp_init     0.0'
        f' --attn_diversity_weight 5.0'
        f' --cls_dropout_p      0.0'
        f' --cls_loss_type      bce'
        f' --lambda1            0.3'
        f' --lambda2            0.2'
        f' --alpha              0.3'
        f' --beta               0.5'
        f' --max_per_class      1000'
        f' --label_smoothing    0.0'
        f' --use_amp'
        f' --grad_checkpoint'
        f' --eval_after_train'
    )
    print("Running:")
    for ln in cmd.split(" --"):
        print("   --" + ln if not ln.startswith("python") else "   " + ln)
    print()
    ret = os.system(cmd)
    if ret != 0:
        print()
        print(f"[ERROR] Pipeline exited with code {ret}. Check logs above.")
    else:
        print()
        print(f"Cell 7 complete - specialist={ACTIVE_MANIPULATION}")
        print(f"Outputs:")
        for f in ["best_model.pth", "ffpp_test_metrics.json",
                  "celebdf_test_metrics.json", "explanation_metrics.json"]:
            path = os.path.join(OUTPUT_DIR, f)
            if os.path.isfile(path):
                print(f"   OK  {f}")
            else:
                print(f"   --  {f} (NOT FOUND)")


---
## Cell 7c — Collapse diagnostic check
Must print **✅ No collapse detected** before proceeding to 10-epoch run.

In [ ]:
import pandas as pd, os
_csv = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(_csv):
    print("metrics.csv not found — did Cell 7 complete?")
else:
    _df = pd.read_csv(_csv, names=["metric","value"], header=0)
    def _get(m):
        row = _df[_df["metric"]==m]["value"]
        return float(row.values[0]) if len(row) else None

    _inter  = _get("inter_sample_cosine_mean")
    _mode   = _get("peak_mode_share")
    _mt_std = _get("m_t_std_mean")

    _warnings = []
    if _inter  is not None and _inter  > 0.95: _warnings.append(f"inter_sample_cosine_mean={_inter:.3f}")
    if _mode   is not None and _mode   > 0.5:  _warnings.append(f"peak_mode_share={_mode:.3f}")
    if _mt_std is not None and _mt_std > 0.13: _warnings.append(f"m_t_std_mean={_mt_std:.3f}")

    if _warnings:
        print("⚠️  EXPLANATION COLLAPSE DETECTED:")
        for w in _warnings: print("   -", w)
        print("Do NOT proceed to 10-epoch run. Diagnose the explanation head first.")
    else:
        print("✅ No collapse detected. Safe to proceed to longer runs.")


---
## Cell 8 — Metrics table

In [ ]:
import os, json, pandas as pd
from IPython.display import display, HTML

# ── FF++ + standard metrics (metrics.csv from evaluate.py) ───────────────────
csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(csv_path):
    print("metrics.csv not found. Did Cell 7 complete without errors?")
else:
    df = pd.read_csv(csv_path, header=0, names=["Metric", "Value"])
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").round(4)

    display(HTML(f"<h3>FF++ Test Metrics — specialist: <code>{ACTIVE_MANIPULATION}</code></h3>"))
    display(df)

    # Verdict
    auc_row = df.loc[df["Metric"] == "auc_roc", "Value"]
    if len(auc_row) and not pd.isna(auc_row.values[0]):
        v = auc_row.values[0]
        if   v >= 0.85: msg = "STRONG"
        elif v >= 0.70: msg = "GOOD"
        elif v >= 0.55: msg = "WEAK"
        else:           msg = "NEAR-CHANCE"
        print(f"AUC-ROC = {v:.4f}  -->  {msg}")

# ── Celeb-DF metrics ─────────────────────────────────────────────────────────
celeb_path = os.path.join(OUTPUT_DIR, "celebdf_test_metrics.json")
if os.path.exists(celeb_path):
    with open(celeb_path) as f:
        celeb = json.load(f)
    rows = [(k, v) for k, v in celeb.items()
            if isinstance(v, (int, float)) and not isinstance(v, bool)]
    cdf = pd.DataFrame(rows, columns=["Metric", "Value"])
    cdf["Value"] = pd.to_numeric(cdf["Value"], errors="coerce").round(4)
    display(HTML(f"<h3>Celeb-DF v2 Test Metrics — transfer from <code>{ACTIVE_MANIPULATION}</code></h3>"))
    display(cdf)
    if "auc_roc" in celeb:
        print(f"Celeb-DF AUC-ROC = {celeb['auc_roc']:.4f}  (FF++->Celeb-DF transfer)")
else:
    print("celebdf_test_metrics.json not found - did --celebdf_eval run?")

# ── Explanation suite ────────────────────────────────────────────────────────
exp_path = os.path.join(OUTPUT_DIR, "explanation_metrics.json")
if os.path.exists(exp_path):
    with open(exp_path) as f:
        exp = json.load(f)
    display(HTML(f"<h3>Explanation Metrics (Intrinsic + Frame-Drop + Stability)</h3>"))
    # Flatten the nested dict for display
    flat = []
    for section, payload in exp.items():
        if isinstance(payload, dict):
            for k, v in payload.items():
                flat.append((f"{section}.{k}", v))
        else:
            flat.append((section, payload))
    edf = pd.DataFrame(flat, columns=["Metric", "Value"])
    # Round numeric values
    def _fmt(x):
        try:
            return round(float(x), 4)
        except Exception:
            return x
    edf["Value"] = edf["Value"].apply(_fmt)
    display(edf)
else:
    print("explanation_metrics.json not found - did --explanation_suite run?")


---
## Cell 9 — Detection graphs (ROC, PR, confusion matrix, score distribution)

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

graphs = [
    ("roc_curve.png",         "ROC Curve"),
    ("pr_curve.png",          "Precision-Recall Curve"),
    ("confusion_matrix.png",  "Confusion Matrix"),
    ("score_distribution.png","Score Distribution"),
]
available = [(f, t) for f, t in graphs
             if os.path.exists(os.path.join(OUTPUT_DIR, f))]

if not available:
    print("No graph files found — did Cell 7 produce output?")
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]
    for ax, (fname, title) in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(OUTPUT_DIR, fname)))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    missing = [t for f, t in graphs if (f, t) not in available]
    if missing:
        print(f"Not yet generated: {missing} — scripts/evaluate.py may need updating.")


---
## Cell 10 — Heatmap videos
Displays all 4 XAI methods (intrinsic, gradcam, rollout, shap) for the first sampled video.

In [ ]:
import os, re
from collections import defaultdict
from IPython.display import display, HTML, Image

heatmap_dir = os.path.join(OUTPUT_DIR, "plots", "heatmaps")
if not os.path.isdir(heatmap_dir):
    print(f"No heatmaps directory at {heatmap_dir}.")
    print("Check that scripts/save_xai_overlays.py ran during Cell 7.")
else:
    methods = ["intrinsic", "gradcam", "rollout"]   # SHAP not generated in Phase 17
    pngs    = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".png"))

    if not pngs:
        print(f"No PNG files in {heatmap_dir}.")
    else:
        # Filename pattern: {video_id}_{label}_conf{prob}_{method}_f{frame_idx}.png
        # Group by video_id, then by method, then by frame
        pat = re.compile(r"(?P<vid>.+?)_(?P<lbl>real|fake)_conf(?P<p>[\d.]+)_(?P<method>\w+)_f(?P<f>\d+)\.png")
        grouped = defaultdict(lambda: defaultdict(list))
        for fname in pngs:
            m = pat.match(fname)
            if not m:
                continue
            vid = m.group("vid")
            method = m.group("method")
            grouped[vid][method].append(fname)

        if not grouped:
            print("Could not parse heatmap filenames. Listing first 5 files:")
            for f in pngs[:5]:
                print(f"   {f}")
        else:
            display(HTML(f"<h3>Heatmap overlays — specialist: <code>{ACTIVE_MANIPULATION}</code></h3>"))
            display(HTML(f"<p>Total videos: {len(grouped)} | Methods: {methods}</p>"))

            # Show ONE sample video, all methods, all frames
            sample_vid = sorted(grouped.keys())[0]
            display(HTML(f"<h4>Sample video: <code>{sample_vid}</code></h4>"))
            for method in methods:
                files = sorted(grouped[sample_vid].get(method, []))
                if not files:
                    print(f"   [{method}] — no files for this video")
                    continue
                display(HTML(f"<p><b>{method.upper()}</b> ({len(files)} frames)</p>"))
                for fname in files:
                    display(Image(os.path.join(heatmap_dir, fname), width=240))

            display(HTML(f"<p style='color:#888'>Remaining {len(grouped)-1} videos available in {heatmap_dir}/</p>"))


---
## Cell 11 — Zip all outputs for download
Download `eahn_outputs.zip` from the Kaggle **Output** tab after this cell.

In [ ]:
import shutil, os

zip_base = f"/kaggle/working/eahn_outputs_{ACTIVE_MANIPULATION}"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
size_mb = os.path.getsize(zip_base + ".zip") / 1e6
print(f"Created  : {zip_base}.zip")
print(f"Size     : {size_mb:.1f} MB")
print(f"Download : Kaggle Output tab -> eahn_outputs_{ACTIVE_MANIPULATION}.zip")
print()
print("After all 5 forks complete, you'll have:")
print("   eahn_outputs_Deepfakes.zip")
print("   eahn_outputs_Face2Face.zip")
print("   eahn_outputs_FaceShifter.zip")
print("   eahn_outputs_FaceSwap.zip")
print("   eahn_outputs_NeuralTextures.zip")
